In [1]:
import os
import json
import cv2
import numpy as np

# ===== PATHS (Phase 2 Training) =====
# ❌ TRASH: Relative paths that break if CWD changes.
# ✅ BULLETPROOF: Absolute paths matching Phase 1 exactly.

BASE_PATH = "/Users/chennupatigunadeep/Downloads/Capstone/DENTEX/training_data/quadrant-enumeration-disease"
IMG_FOLDER = os.path.join(BASE_PATH, "xrays")
JSON_PATH = os.path.join(BASE_PATH, "train_quadrant_enumeration_disease.json")

# Output folder for Phase 2
OUTPUT_BASE = "/Users/chennupatigunadeep/Downloads/Capstone/DENTEX/training_data/quadrant-enumeration-disease/severity_data"

# ===== CREATE FOLDERS =====
train_caries_dir = os.path.join(OUTPUT_BASE, "train", "caries")
train_deep_dir = os.path.join(OUTPUT_BASE, "train", "deep_caries")

os.makedirs(train_caries_dir, exist_ok=True)
os.makedirs(train_deep_dir, exist_ok=True)

# ===== LOAD JSON =====
print(f"Loading annotation from: {JSON_PATH}")
if not os.path.exists(JSON_PATH):
    raise FileNotFoundError(f"CRITICAL ERROR: JSON not found at {JSON_PATH}")

with open(JSON_PATH, "r") as f:
    data = json.load(f)

images = data.get("images", [])
annotations = data.get("annotations", [])

# Map image_id -> file_name
image_id_to_file = {img["id"]: img["file_name"] for img in images}

print(f"Loaded {len(images)} images and {len(annotations)} annotations.")

# ===== HELPER FUNCTION =====
def ann_to_severity_label(ann):
    # Phase 2: Caries vs Deep Caries
    c3 = ann.get("category_id_3", -1)
    if c3 == 1:
        return "caries"
    elif c3 == 3:
        return "deep_caries"
    return None

# ===== PROCESS =====
saved_counts = {"caries": 0, "deep_caries": 0}
skipped = 0
processed = 0

print("Starting processing...")

for idx, ann in enumerate(annotations):
    img_id = ann.get("image_id")
    file_name = image_id_to_file.get(img_id)
    bbox = ann.get("bbox")

    # Basic validity checks
    if file_name is None or bbox is None or len(bbox) != 4:
        skipped += 1
        continue

    # Determine label (Phase 2 logic)
    label = ann_to_severity_label(ann)
    if label is None:
        # Not Caries (1) or Deep Caries (3)
        continue

    # Load Image
    image_path = os.path.join(IMG_FOLDER, file_name)
    if not os.path.exists(image_path):
        skipped += 1
        continue
    img = cv2.imread(image_path)
    if img is None:
        skipped += 1
        continue

    # Robust BBox Logic (Matching Phase 1 EXACTLY)
    x, y, w, h = bbox
    x1 = max(int(np.floor(x)), 0)
    y1 = max(int(np.floor(y)), 0)
    x2 = min(int(np.ceil(x + w)), img.shape[1])
    y2 = min(int(np.ceil(y + h)), img.shape[0])

    if x2 <= x1 or y2 <= y1:
        skipped += 1
        continue

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        skipped += 1
        continue

    # RESIZE (Matching Phase 1: 224x224)
    crop = cv2.resize(crop, (224, 224))

    # Save
    save_dir = train_caries_dir if label == "caries" else train_deep_dir
    filename = f"{img_id}_{ann.get('id', idx)}.png"
    cv2.imwrite(os.path.join(save_dir, filename), crop)

    saved_counts[label] += 1
    processed += 1
    
    if processed % 100 == 0:
        print(f"Processed {processed}...", end="\r")

# ===== SUMMARY =====
print("\nTRAIN DONE")
print("Counts:", saved_counts)
print("Skipped (invalid/empty):", skipped)

Loading annotation from: /Users/chennupatigunadeep/Downloads/Capstone/DENTEX/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json
Loaded 705 images and 3529 annotations.
Starting processing...
Processed 2700...
TRAIN DONE
Counts: {'caries': 2189, 'deep_caries': 578}
Skipped (invalid/empty): 0


In [2]:
import os # Ensure imports exist if cell run independently
import json
import cv2
import numpy as np

# ===== PATHS (Phase 2 Validation) =====
# ❌ TRASH: Relative paths that break if CWD changes.
# ✅ BULLETPROOF: Absolute paths matching Phase 1 exactly.

BASE_PATH = "/Users/chennupatigunadeep/Downloads/Capstone/DENTEX/validation_data/quadrant_enumeration_disease"
IMG_FOLDER = os.path.join(BASE_PATH, "xrays")
JSON_PATH = os.path.join(BASE_PATH, "validation_triple.json")

# Output folder (Redefined here for cell independence)
OUTPUT_BASE = "/Users/chennupatigunadeep/Downloads/Capstone/DENTEX/training_data/quadrant-enumeration-disease/severity_data"
val_caries_dir = os.path.join(OUTPUT_BASE, "val", "caries")
val_deep_dir = os.path.join(OUTPUT_BASE, "val", "deep_caries")

os.makedirs(val_caries_dir, exist_ok=True)
os.makedirs(val_deep_dir, exist_ok=True)

# ===== LOAD JSON =====
print(f"Loading annotation from: {JSON_PATH}")
if not os.path.exists(JSON_PATH):
    raise FileNotFoundError(f"CRITICAL ERROR: JSON not found at {JSON_PATH}")

with open(JSON_PATH, "r") as f:
    data = json.load(f)

images = data.get("images", [])
annotations = data.get("annotations", [])

# Map image_id -> file_name
image_id_to_file = {img["id"]: img["file_name"] for img in images}

print(f"Loaded {len(images)} images and {len(annotations)} annotations.")

# ===== HELPER FUNCTION (Redefined for independence) =====
def ann_to_severity_label(ann):
    # Phase 2: Caries vs Deep Caries
    c3 = ann.get("category_id_3", -1)
    if c3 == 1:
        return "caries"
    elif c3 == 3:
        return "deep_caries"
    return None

# ===== PROCESS =====
saved_counts = {"caries": 0, "deep_caries": 0}
skipped = 0
processed = 0

print("Starting validation processing...")

for idx, ann in enumerate(annotations):
    img_id = ann.get("image_id")
    file_name = image_id_to_file.get(img_id)
    bbox = ann.get("bbox")

    # Basic validity checks
    if file_name is None or bbox is None or len(bbox) != 4:
        skipped += 1
        continue

    # Determine label (Phase 2 logic)
    label = ann_to_severity_label(ann)
    if label is None:
        continue

    # Load Image


    image_path = os.path.join(IMG_FOLDER, file_name)
    if not os.path.exists(image_path):
        skipped += 1
        continue
    img = cv2.imread(image_path)
    if img is None:
        skipped += 1
        continue

    # Robust BBox Logic
    x, y, w, h = bbox
    x1 = max(int(np.floor(x)), 0)
    y1 = max(int(np.floor(y)), 0)
    x2 = min(int(np.ceil(x + w)), img.shape[1])
    y2 = min(int(np.ceil(y + h)), img.shape[0])

    if x2 <= x1 or y2 <= y1:
        skipped += 1
        continue

    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        skipped += 1
        continue

    # RESIZE (Matching Phase 1: 224x224)
    crop = cv2.resize(crop, (224, 224))

    # Save
    save_dir = val_caries_dir if label == "caries" else val_deep_dir
    filename = f"{img_id}_{ann.get('id', idx)}.png"
    cv2.imwrite(os.path.join(save_dir, filename), crop)

    saved_counts[label] += 1
    processed += 1

    if processed % 100 == 0:
        print(f"Processed {processed}...", end="\r")

# ===== SUMMARY =====
print("\nVALIDATION DONE")
print("Counts:", saved_counts)
print("Skipped (invalid/empty):", skipped)

Loading annotation from: /Users/chennupatigunadeep/Downloads/Capstone/DENTEX/validation_data/quadrant_enumeration_disease/validation_triple.json
Loaded 50 images and 182 annotations.
Starting validation processing...
Processed 100...
VALIDATION DONE
Counts: {'caries': 101, 'deep_caries': 32}
Skipped (invalid/empty): 0
